# YOLO11n-seg + SimAM + Coordinate Attention (CA)

This notebook follows the same workflow as the previous notebooks and builds a **YOLO11n instance segmentation model** with **Coordinate Attention (CA) plus SimAM before the Segment head**.

**Experiment goal**
- Baseline: YOLO11n-seg
- Modified: YOLO11n-seg + CA + SimAM
- Task: Shrimp disease instance segmentation


## 1. Environment check

In [ ]:
import os, sys, platform
from pathlib import Path

print('Python:', sys.version)
print('Platform:', platform.platform())

try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as e:
    print('Torch import error:', e)


## 2. Clone Ultralytics and install editable package

In [ ]:
ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
WORK_DIR = ROOT / 'yolov11n_attention'
ULTRA_REPO = WORK_DIR / 'ultralytics'
WORK_DIR.mkdir(parents=True, exist_ok=True)
print('ROOT =', ROOT)
print('WORK_DIR =', WORK_DIR)

if not ULTRA_REPO.exists():
    !git clone https://github.com/ultralytics/ultralytics.git {ULTRA_REPO}
else:
    print('Ultralytics repo already exists:', ULTRA_REPO)

%cd {ULTRA_REPO}
!pip install -e .


## 3. Add Coordinate Attention and SimAM into `conv.py`

In [ ]:
from pathlib import Path

conv_path = ULTRA_REPO / 'ultralytics' / 'nn' / 'modules' / 'conv.py'
text = conv_path.read_text(encoding='utf-8')

coordatt_block = '''

class h_sigmoid(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.relu = nn.ReLU6(inplace=inplace)

    def forward(self, x):
        return self.relu(x + 3) / 6


class h_swish(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.sigmoid = h_sigmoid(inplace=inplace)

    def forward(self, x):
        return x * self.sigmoid(x)


class CoordAtt(nn.Module):
    def __init__(self, c1, c2=None, reduction=32):
        super().__init__()
        c2 = c1 if c2 is None else c2
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        mip = max(8, c1 // reduction)
        self.conv1 = nn.Conv2d(c1, mip, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = h_swish()
        self.conv_h = nn.Conv2d(mip, c2, kernel_size=1, stride=1, padding=0)
        self.conv_w = nn.Conv2d(mip, c2, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        identity = x
        n, c, h, w = x.size()
        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)
        y = torch.cat([x_h, x_w], dim=2)
        y = self.conv1(y)
        y = self.bn1(y)
        y = self.act(y)
        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        a_h = self.conv_h(x_h).sigmoid()
        a_w = self.conv_w(x_w).sigmoid()
        return identity * a_h * a_w
'''

simam_block = '''

class SimAM(nn.Module):
    def __init__(self, c1=None, e_lambda=1e-4):
        super().__init__()
        self.e_lambda = e_lambda
        self.activation = nn.Sigmoid()

    def forward(self, x):
        b, c, h, w = x.size()
        n = h * w - 1
        if n <= 0:
            return x
        x_minus_mu_square = (x - x.mean(dim=[2, 3], keepdim=True)).pow(2)
        y = x_minus_mu_square / (
            4 * (x_minus_mu_square.sum(dim=[2, 3], keepdim=True) / n + self.e_lambda)
        ) + 0.5
        return x * self.activation(y)
'''

updated = text
if 'class CoordAtt(nn.Module):' not in updated:
    updated += coordatt_block
if 'class SimAM(nn.Module):' not in updated:
    updated += simam_block

conv_path.write_text(updated, encoding='utf-8')
print('Updated:', conv_path)


## 4. Export new modules in `__init__.py`

In [ ]:
init_path = ULTRA_REPO / 'ultralytics' / 'nn' / 'modules' / '__init__.py'
text = init_path.read_text(encoding='utf-8')

if 'CoordAtt' not in text:
    if 'from .conv import (' in text:
        text = text.replace('from .conv import (', 'from .conv import (\n    CoordAtt,\n    SimAM,')
    else:
        text += '
from .conv import CoordAtt, SimAM
'

if '__all__' in text and 'CoordAtt' not in text.split('__all__', 1)[1]:
    text = text.replace('"Concat",', '"Concat",\n    "CoordAtt",\n    "SimAM",')

init_path.write_text(text, encoding='utf-8')
print('Updated:', init_path)


## 5. Import/register modules in `tasks.py`

In [ ]:
tasks_path = ULTRA_REPO / 'ultralytics' / 'nn' / 'tasks.py'
text = tasks_path.read_text(encoding='utf-8')

if 'CoordAtt' not in text:
    text = text.replace('from ultralytics.nn.modules import (', 'from ultralytics.nn.modules import (\n    CoordAtt,\n    SimAM,')

anchor = 'elif m in frozenset({Detect, WorldDetect, Segment, Pose, OBB, ImagePoolingAttn, v10Detect}):'
helper = '''elif m is CoordAtt:
            c1 = ch[f]
            c2 = c1
            reduction = args[1] if len(args) > 1 else 32
            args = [c1, c2, reduction]
        elif m is SimAM:
            c2 = ch[f]
            args = [c2, *args]
        '''
if 'elif m is CoordAtt:' not in text and anchor in text:
    text = text.replace(anchor, helper + anchor)

tasks_path.write_text(text, encoding='utf-8')
print('Updated:', tasks_path)


## 6. Create YAML model: `yolo11n-seg-simam-ca-head.yaml`

In [ ]:
cfg_dir = ULTRA_REPO / 'ultralytics' / 'cfg' / 'models' / '11'
base_yaml = cfg_dir / 'yolo11-seg.yaml'
new_yaml = cfg_dir / 'yolo11n-seg-simam-ca-head.yaml'

text = base_yaml.read_text(encoding='utf-8')
old = '- [[16, 19, 22], 1, Segment, [nc, 32, 256]]'
new = '''# Coordinate Attention + SimAM inserted before Segment head for P3, P4, P5 features
  - [16, 1, CoordAtt, [256, 32]]
  - [23, 1, SimAM, []]
  - [19, 1, CoordAtt, [512, 32]]
  - [25, 1, SimAM, []]
  - [22, 1, CoordAtt, [1024, 32]]
  - [27, 1, SimAM, []]
  - [[24, 26, 28], 1, Segment, [nc, 32, 256]]'''

if old in text:
    text = text.replace(old, new)
else:
    print('WARNING: Expected Segment line not found. Please manually inspect base YAML.')

new_yaml.write_text(text, encoding='utf-8')
print('Created:', new_yaml)


## 7. Configure paths and parameters

In [ ]:
from pathlib import Path

MODEL_YAML = ULTRA_REPO / 'ultralytics' / 'cfg' / 'models' / '11' / 'yolo11n-seg-simam-ca-head.yaml'
DATA_YAML = '/path/to/your/data.yaml'  # TODO: update
PROJECT_DIR = 'runs/shrimp_yolo11n_seg_simam_ca'
EXP_NAME = 'yolo11n_seg_simam_ca_head'
IMG_SIZE = 640
EPOCHS = 100
BATCH = 8
DEVICE = 0
WORKERS = 4

print('MODEL_YAML:', MODEL_YAML)
print('DATA_YAML:', DATA_YAML)
print('MODEL EXISTS:', MODEL_YAML.exists())
print('DATA EXISTS:', Path(DATA_YAML).exists())


## 8. Optional Roboflow download block

In [ ]:
# %pip install roboflow -q
# from roboflow import Roboflow
# rf = Roboflow(api_key='YOUR_API_KEY')
# project = rf.workspace('YOUR_WORKSPACE').project('YOUR_PROJECT')
# dataset = project.version(YOUR_VERSION).download('yolov11')
# DATA_YAML = str(Path(dataset.location) / 'data.yaml')
# print('DATA_YAML =', DATA_YAML)


## 9. Import local Ultralytics build

In [ ]:
import sys
sys.path.insert(0, str(ULTRA_REPO))
from ultralytics import YOLO
import ultralytics
print('Ultralytics version:', ultralytics.__version__)


## 10. Test build model before training

In [ ]:
model = YOLO(str(MODEL_YAML))
model.info(verbose=True)
print('Model build successful.')


## 11. Dummy forward test

In [ ]:
import torch

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
model.model.eval()
with torch.no_grad():
    out = model.model(dummy)
print('Forward test successful.')
print(type(out))


## 12. Load pretrained weights

In [ ]:
model = YOLO(str(MODEL_YAML))
try:
    model.load('yolo11n-seg.pt')
    print('Loaded pretrained yolo11n-seg.pt successfully.')
except Exception as e:
    print('Could not fully load pretrained weights:')
    print(e)


## 13. Quick sanity training (3 epochs)

In [ ]:
if not Path(DATA_YAML).exists():
    raise FileNotFoundError(f'Please update DATA_YAML first: {DATA_YAML}')

quick_results = model.train(
    data=DATA_YAML,
    epochs=3,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    project=PROJECT_DIR,
    name=EXP_NAME + '_sanity',
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    close_mosaic=0,
    mask_ratio=4,
    overlap_mask=True
)


## 14. Full training

In [ ]:
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    project=PROJECT_DIR,
    name=EXP_NAME,
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    patience=30,
    close_mosaic=10,
    mask_ratio=4,
    overlap_mask=True
)


## 15. Validate best checkpoint

In [ ]:
BEST_PT = Path(PROJECT_DIR) / EXP_NAME / 'weights' / 'best.pt'
print('BEST_PT:', BEST_PT)

best_model = YOLO(str(BEST_PT))
metrics = best_model.val(data=DATA_YAML, imgsz=IMG_SIZE, device=DEVICE, split='val')
print('Box mAP50:', metrics.box.map50)
print('Box mAP50-95:', metrics.box.map)
if hasattr(metrics, 'seg'):
    print('Mask mAP50:', metrics.seg.map50)
    print('Mask mAP50-95:', metrics.seg.map)


## 16. Predict on test images

In [ ]:
TEST_SOURCE = '/path/to/test/images'  # TODO: update
pred_results = best_model.predict(
    source=TEST_SOURCE,
    imgsz=IMG_SIZE,
    conf=0.25,
    iou=0.5,
    save=True,
    save_txt=True,
    save_conf=True,
    project='runs/predict_shrimp_simam_ca',
    name='simam_ca_predict'
)


## 17. Visualize predictions

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

pred_dir = Path('runs/predict_shrimp_simam_ca/simam_ca_predict')
images = []
for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp'):
    images.extend(sorted(pred_dir.glob(ext)))

if not images:
    print('No prediction images found in:', pred_dir)
else:
    show_n = min(8, len(images))
    fig, axes = plt.subplots(show_n, 1, figsize=(10, 4 * show_n))
    if show_n == 1:
        axes = [axes]
    for ax, img_path in zip(axes, images[:show_n]):
        img = Image.open(img_path)
        ax.imshow(img)
        ax.set_title(img_path.name)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


## 18. Optional export

In [ ]:
best_model.export(format='onnx', imgsz=IMG_SIZE, simplify=True, opset=12)
try:
    best_model.export(format='tflite', imgsz=IMG_SIZE, half=True)
except Exception as e:
    print('TFLite export failed:', e)


## 19. Experiment tracking

| Model | Mask mAP50 | Mask mAP50-95 | Box mAP50 | Params | FLOPs | Latency | FPS | Size |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| YOLO11n-seg baseline | | | | | | | | |
| YOLO11n-seg + SimAM Head | | | | | | | | |
| YOLO11n-seg + CA Head | | | | | | | | |
| YOLO11n-seg + SimAM + CA Head | | | | | | | | |
